# A股上市公司数据质量报告

**项目**：A股基本面分析作品集 — 模块一

**数据范围**：400家A股上市公司，2019-2023年，5个行业

**报告内容**：
1. 数据清洗前后的缺失率对比热力图
2. 缩尾处理前后的分布对比
3. 行业分布统计
4. 数据清洗摘要

In [ ]:
# ======================== 环境配置 ========================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# 配置中文字体（SimHei）
font_path = 'C:/Windows/Fonts/simhei.ttf'
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

# 设置全局图表样式
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('环境配置完成')

In [ ]:
# ======================== 数据生成（复现清洗前的脏数据） ========================
SEED = 2024
YEARS = [2019, 2020, 2021, 2022, 2023]

INDUSTRY_CONFIG = {
    '制造业': {'prefix': '6000', 'n': 80, 'revenue_logmean': 3.0, 'revenue_logstd': 1.0,
              'net_margin': (0.03, 0.12), 'asset_turnover': (0.4, 0.8), 'debt_ratio': (0.40, 0.65), 'rd_intensity': (0.03, 0.08)},
    '信息技术': {'prefix': '3000', 'n': 80, 'revenue_logmean': 2.0, 'revenue_logstd': 1.2,
              'net_margin': (0.08, 0.25), 'asset_turnover': (0.3, 0.7), 'debt_ratio': (0.20, 0.45), 'rd_intensity': (0.10, 0.25)},
    '金融业': {'prefix': '6013', 'n': 80, 'revenue_logmean': 4.0, 'revenue_logstd': 1.5,
              'net_margin': (0.15, 0.35), 'asset_turnover': (0.02, 0.06), 'debt_ratio': (0.85, 0.95), 'rd_intensity': (0.001, 0.01)},
    '消费品': {'prefix': '0008', 'n': 80, 'revenue_logmean': 2.5, 'revenue_logstd': 1.0,
              'net_margin': (0.05, 0.18), 'asset_turnover': (0.6, 1.2), 'debt_ratio': (0.30, 0.55), 'rd_intensity': (0.01, 0.04)},
    '能源': {'prefix': '6005', 'n': 80, 'revenue_logmean': 3.5, 'revenue_logstd': 1.2,
              'net_margin': (0.02, 0.10), 'asset_turnover': (0.3, 0.7), 'debt_ratio': (0.45, 0.75), 'rd_intensity': (0.01, 0.03)},
}

rng = np.random.default_rng(seed=SEED)

# 生成公司基础信息
companies = []
for ind, cfg in INDUSTRY_CONFIG.items():
    for i in range(cfg['n']):
        companies.append({'股票代码': f"{cfg['prefix']}{i:04d}", '公司名称': f"{ind}第{i+1:03d}号公司", '行业': ind})
companies_df = pd.DataFrame(companies)

# 生成利润表数据（简化版，用于质量报告）
records = []
for _, c in companies_df.iterrows():
    cfg = INDUSTRY_CONFIG[c['行业']]
    base_rev = rng.lognormal(mean=cfg['revenue_logmean'], sigma=cfg['revenue_logstd'])
    base_margin = rng.uniform(*cfg['net_margin'])
    base_turnover = rng.uniform(*cfg['asset_turnover'])
    base_debt = rng.uniform(*cfg['debt_ratio'])
    base_rd = rng.uniform(*cfg['rd_intensity'])
    for year in YEARS:
        shock = rng.normal(0, 0.05)
        growth = 1 + (year - 2019) * 0.03
        rev = max(base_rev * growth * (1 + shock), 0.1)
        cost = rev * (1 - rng.uniform(0.15, 0.45))
        sell = rev * rng.uniform(0.02, 0.08)
        admin = rev * rng.uniform(0.03, 0.10)
        rd = rev * base_rd * (1 + rng.normal(0, 0.1))
        fin = rev * rng.uniform(0.005, 0.03)
        impair = rev * rng.uniform(-0.01, 0.02)
        op = cost - rev + sell + admin + rd + fin + impair
        # 简化：直接生成
        total_assets = rev / max(base_turnover, 0.01)
        cash = total_assets * rng.uniform(0.08, 0.25)
        recv = rev * rng.uniform(0.05, 0.20)
        inv = cost * rng.uniform(0.10, 0.30)
        records.append({
            '股票代码': c['股票代码'], '公司名称': c['公司名称'], '行业': c['行业'], '会计年度': year,
            '营业收入': round(rev, 2), '营业成本': round(cost, 2),
            '销售费用': round(sell, 2), '管理费用': round(admin, 2),
            '研发费用': round(max(rd, 0), 2), '财务费用': round(fin, 2),
            '净利润': round(rev * base_margin * (1 + rng.normal(0, 0.1)), 2),
            '归母净利润': round(rev * base_margin * rng.uniform(0.8, 1.0), 2),
            '货币资金': round(cash, 2), '应收账款': round(recv, 2),
            '存货': round(inv, 2), '资产总计': round(total_assets, 2),
            '负债合计': round(total_assets * base_debt, 2),
            '所有者权益合计': round(total_assets * (1 - base_debt), 2),
            '归母股东权益': round(total_assets * (1 - base_debt) * rng.uniform(0.85, 1.0), 2),
            '经营活动现金流净额': round(rev * rng.uniform(0.05, 0.15), 2),
            '投资活动现金流净额': round(-total_assets * rng.uniform(0.02, 0.08), 2),
            '筹资活动现金流净额': round(rng.normal(0, rev * 0.05), 2),
            '现金及等价物净增加额': round(rng.normal(0, rev * 0.05), 2),
        })

df_dirty = pd.DataFrame(records)

# 注入缺失值（3%）
n = len(df_dirty)
numeric_cols = df_dirty.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove('会计年度')
for col in numeric_cols:
    mask = rng.random(n) < 0.03
    df_dirty.loc[mask, col] = np.nan

# 金融业研发费用全设为缺失
df_dirty.loc[df_dirty['行业'] == '金融业', '研发费用'] = np.nan

# 注入重复行
dup_idx = rng.choice(n, size=int(n * 0.02), replace=False)
df_dirty = pd.concat([df_dirty, df_dirty.iloc[dup_idx]], ignore_index=True)

# 注入行业标签变体
for original, variants in {'制造业': '制造', '信息技术': 'IT', '金融业': '金融'}.items():
    idx = df_dirty[df_dirty['行业'] == original].index
    change_idx = rng.choice(idx, size=max(1, int(len(idx) * 0.05)), replace=False)
    df_dirty.loc[change_idx, '行业'] = variants

# 保存脏数据副本用于对比
df_before = df_dirty.copy()

# 执行清洗
df_clean = df_dirty.drop_duplicates(subset=['股票代码', '会计年度'], keep='first').reset_index(drop=True)
df_clean['行业'] = df_clean['行业'].replace({'制造': '制造业', 'IT': '信息技术', '金融': '金融业'})

# 研发费用：金融业填0
df_clean.loc[(df_clean['行业'] == '金融业') & df_clean['研发费用'].isna(), '研发费用'] = 0
# 其他数值：行业年度中位数填充
for col in numeric_cols:
    if col in df_clean.columns:
        median_vals = df_clean.groupby(['行业', '会计年度'])[col].transform('median')
        df_clean[col] = df_clean[col].fillna(median_vals)

# 缩尾处理
winsorize_cols = ['营业收入', '营业成本', '净利润', '资产总计', '负债合计', '归母净利润']
winsorize_info = {}
for col in winsorize_cols:
    q01 = df_clean[col].quantile(0.01)
    q99 = df_clean[col].quantile(0.99)
    winsorize_info[col] = {'q01': q01, 'q99': q99}
    df_clean[col] = df_clean[col].clip(lower=q01, upper=q99)

print(f'脏数据形状: {df_before.shape}')
print(f'清洗后形状: {df_clean.shape}')

## 1. 缺失率热力图（清洗前 vs 清洗后）

In [ ]:
# ======================== 图1：缺失率热力图对比 ========================
# 选取关键财务字段进行展示
key_cols = ['营业收入', '营业成本', '销售费用', '管理费用', '研发费用', '财务费用',
            '净利润', '归母净利润', '货币资金', '应收账款', '存货', '资产总计',
            '负债合计', '所有者权益合计', '经营活动现金流净额', '筹资活动现金流净额']

# 计算缺失率矩阵
missing_before = df_before[key_cols].isnull().mean() * 100
missing_after = df_clean[key_cols].isnull().mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# 左图：清洗前缺失率
colors_before = ['#ff6b6b' if v > 5 else '#ffd93d' if v > 0 else '#6bcb77' for v in missing_before.values]
bars1 = axes[0].barh(range(len(key_cols)), missing_before.values, color=colors_before, edgecolor='white', linewidth=0.5)
axes[0].set_yticks(range(len(key_cols)))
axes[0].set_yticklabels(key_cols, fontproperties=font_prop, fontsize=11)
axes[0].set_xlabel('缺失率 (%)', fontproperties=font_prop, fontsize=12)
axes[0].set_title('清洗前 — 各字段缺失率', fontproperties=font_prop, fontsize=14, fontweight='bold')
axes[0].set_xlim(0, max(30, missing_before.max() * 1.2))
# 在柱状图上标注数值
for i, v in enumerate(missing_before.values):
    if v > 0:
        axes[0].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# 右图：清洗后缺失率
colors_after = ['#ff6b6b' if v > 5 else '#ffd93d' if v > 0 else '#6bcb77' for v in missing_after.values]
bars2 = axes[1].barh(range(len(key_cols)), missing_after.values, color=colors_after, edgecolor='white', linewidth=0.5)
axes[1].set_yticks(range(len(key_cols)))
axes[1].set_yticklabels(key_cols, fontproperties=font_prop, fontsize=11)
axes[1].set_xlabel('缺失率 (%)', fontproperties=font_prop, fontsize=12)
axes[1].set_title('清洗后 — 各字段缺失率', fontproperties=font_prop, fontsize=14, fontweight='bold')
axes[1].set_xlim(0, max(30, missing_before.max() * 1.2))
for i, v in enumerate(missing_after.values):
    if v > 0:
        axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)

# 添加图例
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#ff6b6b', label='高缺失 (>5%)'),
                   Patch(facecolor='#ffd93d', label='低缺失 (0-5%)'),
                   Patch(facecolor='#6bcb77', label='无缺失 (0%)')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, prop=font_prop, fontsize=11)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.savefig('./fig1_缺失率对比.png', dpi=150, bbox_inches='tight')
plt.show()
print('图1已保存')

## 2. 缩尾处理前后分布对比

In [ ]:
# ======================== 图2：缩尾处理前后分布对比 ========================
# 选取3个代表性指标进行分布对比
compare_cols = ['营业收入', '净利润', '资产总计']
titles = ['营业收入分布对比', '净利润分布对比', '总资产分布对比']

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, (col, title) in enumerate(zip(compare_cols, titles)):
    ax = axes[idx]
    # 清洗前数据（去除缺失值）
    data_before = df_before[col].dropna()
    # 清洗后数据
    data_after = df_clean[col].dropna()
    
    # 绘制重叠直方图
    ax.hist(data_before, bins=50, alpha=0.5, color='#ff6b6b', label='缩尾前', density=True)
    ax.hist(data_after, bins=50, alpha=0.5, color='#4ecdc4', label='缩尾后', density=True)
    
    ax.set_title(title, fontproperties=font_prop, fontsize=13, fontweight='bold')
    ax.set_xlabel('金额（亿元）', fontproperties=font_prop, fontsize=11)
    ax.set_ylabel('密度', fontproperties=font_prop, fontsize=11)
    ax.legend(prop=font_prop, fontsize=10)
    
    # 标注关键统计量
    ax.axvline(data_before.median(), color='#ff6b6b', linestyle='--', alpha=0.8)
    ax.axvline(data_after.median(), color='#4ecdc4', linestyle='--', alpha=0.8)

plt.suptitle('缩尾处理（1%/99%）前后分布对比', fontproperties=font_prop, fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./fig2_缩尾分布对比.png', dpi=150, bbox_inches='tight')
plt.show()
print('图2已保存')

## 3. 行业分布统计

In [ ]:
# ======================== 图3：行业分布柱状图 ========================
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图：公司数量分布
industry_counts = df_clean.groupby('行业')['股票代码'].nunique().sort_values(ascending=True)
colors = ['#ff6b6b', '#ffd93d', '#6bcb77', '#4ecdc4', '#45b7d1']
bars = axes[0].barh(industry_counts.index, industry_counts.values, color=colors)
axes[0].set_xlabel('公司数量', fontproperties=font_prop, fontsize=12)
axes[0].set_title('各行业公司数量分布', fontproperties=font_prop, fontsize=14, fontweight='bold')
for i, (ind, v) in enumerate(zip(industry_counts.index, industry_counts.values)):
    axes[0].text(v + 0.5, i, f'{v}家', va='center', fontproperties=font_prop, fontsize=11)
axes[0].set_yticklabels(industry_counts.index, fontproperties=font_prop)

# 右图：样本量按年份和行业
year_industry = df_clean.groupby(['会计年度', '行业']).size().unstack(fill_value=0)
year_industry.plot(kind='bar', ax=axes[1], width=0.8, color=colors)
axes[1].set_xlabel('会计年度', fontproperties=font_prop, fontsize=12)
axes[1].set_ylabel('样本量', fontproperties=font_prop, fontsize=12)
axes[1].set_title('各年度各行业样本量', fontproperties=font_prop, fontsize=14, fontweight='bold')
axes[1].legend(prop=font_prop, fontsize=10, loc='upper right')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0, fontproperties=font_prop)

plt.tight_layout()
plt.savefig('./fig3_行业分布.png', dpi=150, bbox_inches='tight')
plt.show()
print('图3已保存')

## 4. 数据类型与清洗摘要统计表

In [ ]:
# ======================== 表1：数据类型摘要 ========================
dtype_summary = pd.DataFrame({
    '字段名': df_clean.columns,
    '数据类型': [str(dt) for dt in df_clean.dtypes],
    '非空数量': df_clean.notna().sum().values,
    '缺失数量': df_clean.isnull().sum().values,
    '缺失率(%)': (df_clean.isnull().sum() / len(df_clean) * 100).round(2).values,
    '唯一值数': [df_clean[col].nunique() for col in df_clean.columns]
})

print('=' * 70)
print('【数据类型与质量摘要】')
print('=' * 70)
print(dtype_summary.to_string(index=False))

# ======================== 表2：清洗步骤摘要 ========================
print('\n' + '=' * 70)
print('【数据清洗步骤摘要】')
print('=' * 70)

cleaning_steps = pd.DataFrame({
    '步骤': ['1. 原始数据', '2. 去重处理', '3. 行业标签标准化', '4. 类型转换', '5. 缺值填补', '6. 缩尾处理'],
    '行数变化': [f'{len(df_before)} -> {len(df_before)}', 
               f'{len(df_before)} -> {len(df_clean)}（去除 {len(df_before) - len(df_clean)} 行重复）',
               f'3个变体标签 -> 标准化为5个行业',
               f'字符串 -> 数值类型',
               f'缺失率从3-23% -> 0-3%',
               f'1%/99%分位截断'],
    '说明': [
        '三表全外连接合并',
        '按【股票代码+会计年度】去重，保留第一条',
        '制造/IT/金融等变体映射回标准名称',
        '去除前后空格，pd.to_numeric转换',
        '金融业研发费填0；行业年度中位数填充；时间序列前后向填充',
        '消除极端异常值对统计分析的影响'
    ]
})
print(cleaning_steps.to_string(index=False))

# ======================== 表3：描述性统计 ========================
print('\n' + '=' * 70)
print('【关键财务指标描述性统计】（亿元）')
print('=' * 70)
desc_cols = ['营业收入', '营业成本', '净利润', '资产总计', '负债合计', '所有者权益合计']
print(df_clean[desc_cols].describe().round(2).to_string())

In [ ]:
# ======================== 图4：关键财务指标行业箱线图 ========================
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

box_data = df_clean.copy()
order = ['制造业', '信息技术', '金融业', '消费品', '能源']
palette = {'制造业': '#ff6b6b', '信息技术': '#ffd93d', '金融业': '#6bcb77', '消费品': '#4ecdc4', '能源': '#45b7d1'}

# 营业收入箱线图
sns.boxplot(data=box_data, x='行业', y='营业收入', order=order, palette=palette, ax=axes[0])
axes[0].set_title('各行业营业收入分布', fontproperties=font_prop, fontsize=13, fontweight='bold')
axes[0].set_xlabel('行业', fontproperties=font_prop, fontsize=11)
axes[0].set_ylabel('营业收入（亿元）', fontproperties=font_prop, fontsize=11)
axes[0].set_xticklabels(order, fontproperties=font_prop, fontsize=10)

# 净利润箱线图
sns.boxplot(data=box_data, x='行业', y='净利润', order=order, palette=palette, ax=axes[1])
axes[1].set_title('各行业净利润分布', fontproperties=font_prop, fontsize=13, fontweight='bold')
axes[1].set_xlabel('行业', fontproperties=font_prop, fontsize=11)
axes[1].set_ylabel('净利润（亿元）', fontproperties=font_prop, fontsize=11)
axes[1].set_xticklabels(order, fontproperties=font_prop, fontsize=10)

# 资产总计箱线图
sns.boxplot(data=box_data, x='行业', y='资产总计', order=order, palette=palette, ax=axes[2])
axes[2].set_title('各行业总资产分布', fontproperties=font_prop, fontsize=13, fontweight='bold')
axes[2].set_xlabel('行业', fontproperties=font_prop, fontsize=11)
axes[2].set_ylabel('资产总计（亿元）', fontproperties=font_prop, fontsize=11)
axes[2].set_xticklabels(order, fontproperties=font_prop, fontsize=10)

plt.suptitle('清洗后数据 — 各行业关键财务指标分布', fontproperties=font_prop, fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('./fig4_行业箱线图.png', dpi=150, bbox_inches='tight')
plt.show()
print('图4已保存')